# Chapter 7 - Lab 5: Parallel multi-account analysis (customer-care edition)

**Goal**: run the Lab 1 pipeline for a whole **portfolio of enterprise accounts** concurrently with
`asyncio.gather`, and measure the speed-up versus a sequential run. This is how a customer-success team
would refresh account health every Monday morning before the QBR (quarterly business review) cycle.

*Finance edition*: five companies (AAPL, MSFT, GOOGL, NVDA, JPM). *Customer-care edition*: five accounts.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared helpers

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%207/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (
    SERVICE_QUALITY_THRESHOLDS, RELATIONSHIP_RISK_THRESHOLDS,
    format_threshold_prompt, stub_lookup,
)
print("Thresholds and stub data loaded.")

## 4. Tools and a pipeline factory

In [ ]:
from llama_index.core.workflow import Context


async def get_account_metrics(ctx: Context, account_id: str) -> str:
    """Used by AccountDataAgent. Reads the account's metrics into shared state."""
    data = stub_lookup(account_id)
    state = await ctx.store.get("state")
    if state["account_id"] == "":
        state["account_id"] = account_id
    state["metrics"] = {**data["service_quality"], **data["relationship_risk"]}
    await ctx.store.set("state", state)
    return f"Metrics extracted for {account_id}."


async def get_service_quality_metrics(ctx: Context) -> str:
    """Used by ServiceQualityAgent. Selects the service-quality metrics and their thresholds."""
    state = await ctx.store.get("state")
    if not state["metrics"]:
        return "No metrics - run get_account_metrics first."
    state["service_quality_metrics"] = {k: state["metrics"][k] for k in SERVICE_QUALITY_THRESHOLDS}
    state["threshold_service_quality_comments"] = format_threshold_prompt(
        SERVICE_QUALITY_THRESHOLDS, "Service Quality")
    await ctx.store.set("state", state)
    return (f"Service-quality metrics for {state['account_id']}: {state['service_quality_metrics']}\n"
            f"{state['threshold_service_quality_comments']}")


async def get_relationship_risk_metrics(ctx: Context) -> str:
    """Used by RelationshipRiskAgent. Selects the relationship-risk metrics and their thresholds."""
    state = await ctx.store.get("state")
    if not state["metrics"]:
        return "No metrics - run get_account_metrics first."
    state["relationship_risk_metrics"] = {k: state["metrics"][k] for k in RELATIONSHIP_RISK_THRESHOLDS}
    state["threshold_relationship_risk_comments"] = format_threshold_prompt(
        RELATIONSHIP_RISK_THRESHOLDS, "Relationship Risk")
    await ctx.store.set("state", state)
    return (f"Relationship-risk metrics for {state['account_id']}: {state['relationship_risk_metrics']}\n"
            f"{state['threshold_relationship_risk_comments']}")


async def get_overall_comments(ctx: Context) -> str:
    """Used by SupervisorAgent. Confirms both scorings are done before the final verdict."""
    state = await ctx.store.get("state")
    if state["threshold_service_quality_comments"] is None:
        return "Service-quality scoring not done."
    if state["threshold_relationship_risk_comments"] is None:
        return "Relationship-risk scoring not done."
    state["overall_comments"] = "Synthesis complete."
    await ctx.store.set("state", state)
    return "Both scorings are complete; write the overall account-health verdict."

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow, FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o", temperature=0)


def build_pipeline() -> AgentWorkflow:
    data = FunctionAgent(
        name="AccountDataAgent", description="Fetches all account metrics into shared state.",
        system_prompt=("Call get_account_metrics exactly once. When it succeeds, immediately call the handoff tool "
                       "to pass control to ServiceQualityAgent. Do not describe the handoff in text: actually call "
                       "the tool. Never produce a final answer yourself."),
        llm=llm, tools=[get_account_metrics], can_handoff_to=["ServiceQualityAgent"],
    )
    quality = FunctionAgent(
        name="ServiceQualityAgent", description="Scores service-quality metrics against thresholds.",
        system_prompt=("Call get_service_quality_metrics exactly once and score each metric from 1 to 10. Then "
                       "immediately call the handoff tool to pass control to RelationshipRiskAgent. Never produce "
                       "a final answer yourself."),
        llm=llm, tools=[get_service_quality_metrics], can_handoff_to=["RelationshipRiskAgent"],
    )
    risk = FunctionAgent(
        name="RelationshipRiskAgent", description="Scores relationship-risk metrics against thresholds.",
        system_prompt=("Call get_relationship_risk_metrics exactly once and score each metric from 1 to 10. Then "
                       "immediately call the handoff tool to pass control to SupervisorAgent. Never produce a "
                       "final answer yourself."),
        llm=llm, tools=[get_relationship_risk_metrics], can_handoff_to=["SupervisorAgent"],
    )
    # can_handoff_to=[] makes the Supervisor terminal.
    supervisor = FunctionAgent(
        name="SupervisorAgent", description="Writes the final account-health verdict.",
        system_prompt=("Call get_overall_comments exactly once, then write the final verdict yourself, starting "
                       "with one of HEALTHY | WATCH | AT RISK | CRITICAL, followed by each metric's 1-10 score. "
                       "You are the last agent: do not hand off."),
        llm=llm, tools=[get_overall_comments], can_handoff_to=[],
    )
    return AgentWorkflow(agents=[data, quality, risk, supervisor], root_agent="AccountDataAgent",
                         initial_state={
        "metrics": {},
        "account_id": "",
        "service_quality_metrics": {},
        "relationship_risk_metrics": {},
        "threshold_service_quality_comments": None,
        "threshold_relationship_risk_comments": None,
        "overall_comments": None,
    })


async def analyze(account_id: str) -> str:
    handler = build_pipeline().run(user_msg=f"Assess the account health of {account_id}.", max_iterations=60)
    return str(await handler)

## 5. Sequential baseline

In [ ]:
import asyncio, time

PORTFOLIO = ["AC-HOTELS", "AC-SCHOOLS", "AC-CLINICS", "AC-COWORK", "AC-RETAIL"]

start = time.time()
sequential_results = {}
for a in PORTFOLIO:
    sequential_results[a] = await analyze(a)
sequential_elapsed = time.time() - start
print(f"Sequential: {sequential_elapsed:.1f}s for {len(PORTFOLIO)} accounts.")

## 6. Parallel with `asyncio.gather`

Each account gets its own pipeline instance. Most time is spent waiting for the LLM, so the runs overlap almost completely. `return_exceptions=True` keeps one failed account from killing the batch.

In [ ]:
start = time.time()
outputs = await asyncio.gather(*(analyze(a) for a in PORTFOLIO), return_exceptions=True)
parallel_results = dict(zip(PORTFOLIO, outputs))
parallel_elapsed = time.time() - start
print(f"Parallel:   {parallel_elapsed:.1f}s for {len(PORTFOLIO)} accounts.")
print(f"Speed-up:   {sequential_elapsed / parallel_elapsed:.1f}x")

## 7. Compare results

In [ ]:
print(f"{'Account':<12} {'Sequential (60 chars)':<62} {'Parallel (60 chars)'}")
for a in PORTFOLIO:
    seq = sequential_results[a][:60].replace("\n", " ")
    par = parallel_results[a]
    par_str = f"ERROR: {par}" if isinstance(par, Exception) else str(par)[:60].replace("\n", " ")
    print(f"{a:<12} {seq:<62} {par_str}")

## 8. Results

* **Speed-up close to the number of accounts**: the work is network-bound (waiting for the LLM).
* **Independence is the prerequisite**: each account's analysis is self-contained.
* **Rate limits matter**: N parallel pipelines mean N times the tokens per second. Chapter 12 sizes
  the fan-out with a semaphore.
* **Combine patterns**: put Lab 4's evaluator inside each parallel pipeline.

## Discussion
Rank the five accounts by urgency. Which two would you put in front of the VP of Customer Success this week, and why?